In [ ]:
import pandas as pd

In [ ]:
pwd()

In [ ]:
df = pd.read_csv(r"..\results\fit_analysis_summary.csv")

In [ ]:
df.head()

In [ ]:
#df['fix_time']=
df['date'] =  pd.to_datetime(df.session_root.str.split('_').str[0], format='%Y%m%d')

In [ ]:
df['fix_time']=[int(k.lower().split('form')[1][:2]) if k.lower().find('form')>-1 else 0 for k in df.filename.values ]

In [ ]:
df.columns

In [ ]:
df['poc'] = df.filename.str.split('_poc').str[1].str.split('_').str[0]
df['pockelcell'] = df.poc.str.replace('p','.').astype(float)

In [ ]:
df['PC'] = ['High' if k>0.3 else 'Low' for k in df.pockelcell.values]
df.PC.value_counts()

# data describe

In [ ]:
df.groupby(['fixation_type', 'em_filter_nm', 'date','cell_type','fix_time', 'PC']).size()

In [ ]:
df.session_root.unique(), df.cell_type.unique()

In [ ]:
import seaborn as sns

In [ ]:
sns.catplot(x='session_root',y='amp_ratio_mean', data=df.loc[(df.em_filter_nm==457)], hue='cell_type',col='fixation_type')

In [ ]:
sns.catplot(x='session_root',y='amp_ratio_mean', data=df.loc[(df.em_filter_nm==457) & (df.PC == "Low") ], hue='cell_type',col='fixation_type')

In [ ]:
df.loc[df.em_filter_nm==457].shape

## a1/a2 median

In [ ]:
sns.catplot(x='session_root',y='amp_ratio_median', data=df.loc[df.em_filter_nm==457], hue='cell_type',col='fixation_type')

In [ ]:
df.columns

In [ ]:
sns.catplot(x='date',y='tau_mean_median_ps', data=df.loc[df.em_filter_nm==457], hue='cell_type',col='fixation_type')

## taumean

In [ ]:
sns.catplot(x='date',y='tau_mean_mean_ps', data=df.loc[df.em_filter_nm==457], hue='cell_type',col='fixation_type',kind='box')

## two pockel cell data filter

In [ ]:
sns.catplot(x='date',y='tau_mean_mean_ps', data=df.loc[df.em_filter_nm==457], hue='cell_type',col='fixation_type',kind='box',row='PC')

In [ ]:
sns.catplot(x='date',y='tau_mean_median_ps', data=df.loc[(df.em_filter_nm==457)&(df.PC=='Low')], hue='cell_type',col='fixation_type')

In [ ]:
sns.catplot(x='date',y='amp_ratio_mean', data=df.loc[(df.em_filter_nm==457)&(df.PC=='Low')], hue='cell_type',col='fixation_type')

# reading asc

In [ ]:
filenames = df.loc[df.em_filter_nm==457]['filename'].unique()
len(filenames)

In [ ]:
%%time
from pathlib import Path
sdtfiles = list(Path(r"E:\18_RK_Circadian\data\raw").rglob('*.sdt'))
len(sdtfiles)

In [ ]:
for filename in filenames:
    #print(filename)
    corr_file_path = [k for k in sdtfiles if k.name == filename]
    
    assert len(corr_file_path)==1, f"Expected exactly one match for {filename}, found {len(corr_file_path)}"

In [ ]:
fn = Path(filename)
[k for k in list(corr_file_path[0].parent.rglob('*.asc')) if (k.name.startswith(fn.stem))&(k.name.endswith('photons.asc'))]

In [ ]:
import numpy as np

In [ ]:
for filename in filenames:
    #print(filename)
    corr_file_path = [k for k in sdtfiles if k.name == filename]
    assert len(corr_file_path)==1, f"Expected exactly one match for {filename}, found {len(corr_file_path)}"
    fn = Path(filename)
    asc_ph_files = [k for k in list(corr_file_path[0].parent.rglob('*.asc')) if (k.name.startswith(fn.stem))&(k.name.endswith('photons.asc'))]
    szfiles = [k for k in asc_ph_files if k.parts[-2].find('-sz-')>-1]
    for szfile in szfiles:
        shiftfile = str(szfile).replace('photons.asc','shift.asc') 
        shift_data = np.loadtxt(shiftfile)
        if shift_data.sum()>0:
            print(szfile.name,  shift_data.sum())
        #break
    #break

### SPC IMAGE 8.9.85 keeps the shift to zero on MLE if you set the checkbox as fixed

In [ ]:
fn_dict={}
for filename in filenames:
    #print(filename)
    corr_file_path = [k for k in sdtfiles if k.name == filename]
    assert len(corr_file_path)==1, f"Expected exactly one match for {filename}, found {len(corr_file_path)}"
    fn = Path(filename)
    asc_ph_files = [k for k in list(corr_file_path[0].parent.rglob('*.asc')) if (k.name.startswith(fn.stem))&(k.name.endswith('photons.asc'))]
    szfiles = [k for k in asc_ph_files if k.parts[-2].find('fitet-sz-c2')>-1]
    fn_dict[filename]=szfiles[-1]

In [ ]:
fn_dict[filename].name

In [ ]:
from matplotlib.pyplot import *

In [ ]:
from skimage.filters import threshold_local

In [ ]:
data_dict = {}
for filename in fn_dict:
    #print(filename, fn_dict[filename].parent)
    ph_file = np.loadtxt(fn_dict[filename])
    color_coded_file = Path(str(fn_dict[filename]).replace('photons.asc','color coded value.asc'))
    if color_coded_file.exists():
        taumean_file = np.loadtxt(color_coded_file)
    else:
        taumean_file = np.array([np.nan])
    a1_file = Path(str(fn_dict[filename]).replace('photons.asc','a1.asc')) 
    if a1_file.exists():
        a1_file = np.loadtxt(a1_file)
    else:
        a1_file = np.array([np.nan])
    a2_file = Path(str(fn_dict[filename]).replace('photons.asc', 'a2.asc'))
    if a2_file.exists():
        a2_file = np.loadtxt(a2_file)
    else:
        a2_file = np.array([np.nan])
    #a1a2 = a1_file/a2_file
    chi_file = Path(str(fn_dict[filename]).replace('photons.asc', 'chi.asc'))
    if chi_file.exists():
        chi_file = np.loadtxt(chi_file)
    else:
        chi_file = np.array([np.nan])
    ma = ph_file > ph_file.mean()*1.2
    #mask = threshold_local(ph_file, block_size=11, offset=8)
    #imshow(ma)
    data_dict[filename] = {
        'filepath':fn_dict[filename],
        'ph':ph_file[ma].mean(), 
        'taumean':taumean_file[ma].mean(),
        'a1':a1_file[ma].mean(),
        'a2':a2_file[ma].mean(),
        'chi':chi_file[ma].mean()}
    
    make_fig()
    output = Path(r'E:\18_RK_Circadian\data\processed_temp\preview') / (fn_dict[filename].stem + '_preview.png')
    savefig(output)
    close()
    
    #break

In [ ]:
#ph_file[ma].mean(), ph_file[~ma].mean(), a1_file[ma].mean(), a1_file[~ma].mean()

In [ ]:
def make_fig():
    figure(figsize=(25,3))
    subplot(1,6,1)
    imshow(ph_file)
    title(f'{ph_file[ma].mean():.1f} vs {ph_file[~ma].mean():.1f}')
    colorbar()
    subplot(1,6,2) 
    imshow(taumean_file)
    title(f'{taumean_file[ma].mean():.1f} vs {taumean_file[~ma].mean():.1f}')
    colorbar()
    subplot(1,6 ,3)  
    imshow(a1_file)
    title(f'{a1_file[ma].mean():.1f} vs {a1_file[~ma].mean():.1f}')
    colorbar()
    subplot(1,6,4)  
    imshow(a2_file)
    title(f'{a2_file[ma].mean():.1f} vs {a2_file[~ma].mean():.1f}')
    colorbar()
    subplot(1,6,5)
    imshow(chi_file)
    title(f'{chi_file[ma].mean():.1f} vs {chi_file[~ma].mean():.1f}')
    colorbar()
    subplot(1,6,6)
    imshow(ma)
    title(f'Mask: {ma.sum()} pixels')
    colorbar()



In [ ]:
df1 = pd.DataFrame.from_dict(data_dict, orient='index')

In [ ]:
df1.reset_index(inplace=True)

In [ ]:
df1.columns

In [ ]:
df1.columns = ['filename', 'filepath', 'ph_mean', 'taumean_mean', 'a1_mean', 'a2_mean', 'chi_mean']

In [ ]:
df1.head()

In [ ]:
df1['datex'] = [k.parts[4].split('_')[0] for k in df1.filepath.values]
df1.datex.value_counts()

In [ ]:
df1['fixation'] = ['L' if k.lower().find('live')>-1 else 'F' for k in df1.filename.values] 
df1.fixation.value_counts()

In [ ]:
df1['fixation'] = ['L' if k.lower().find('live')>-1 else 'F' for k in df1.filename.values] 
df1.fixation.value_counts()

In [ ]:
df1['celltype'] = [k.split('_')[1][0] for k in df1.filename.values] 
df1.celltype.value_counts()

In [ ]:
df1.head()

In [ ]:
sns.catplot(x='datex',y='taumean_mean', data=df1, hue='celltype',col='fixation',kind='box')

In [ ]:
df1['a1bya2'] = df1.a1_mean/df1.a2_mean

In [ ]:
sns.catplot(x='datex',y='a1bya2', data=df1, hue='celltype',col='fixation',kind='strip')

In [ ]:
## 8-cell fits

In [ ]:
Cells